# Preparation of results for F2bc -- learning curve

This notebook summarizes the data computed by ../scripts/F2bc.sh and ../scripts/F2a_gather_results.sh.

In [34]:
import os
import re
import cv2
import sys
import json
import einops
import itertools
import numpy as np
import pandas as pd
import pickle as pkl
import os.path as op
sys.path.append("..")
from tqdm import tqdm
from Bio.Seq import Seq
import Levenshtein as lev
from Bio import pairwise2
import multiprocessing as mp
from itertools import product
from Bio import SeqIO, motifs
from itertools import product
import matplotlib.pyplot as plt
import src.swiper_cli as swiper
from matplotlib.patches import Patch
import src.convergence_test as convergence
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import matthews_corrcoef, accuracy_score, balanced_accuracy_score

In [35]:
import warnings
warnings.filterwarnings('ignore')

In [36]:
class Result():
    """Class for constructing paths for learning curve results and retrieving metrics.

    Attributes:
        BASE (str): The base directory path.
        METRICS (str): The filename for the metrics TSV.
        columns (list): Column names for the resulting DataFrame.
    """

    def __init__(self, base="../data/interim/fig2/ORTHOLOG/LIBRARY/passive/"):
        self.BASE = base
        self.METRICS = "TTS_SEED.metrics.tsv"
        self.columns = ["ortholog", "library", "seed", "tts", "acc", "balacc", "mcc"]

    def make_path(self, ortholog, library, seed, tts):
        """
        Builds the file path for the metrics file.

        Replaces the placeholders in the base path and filename with the
        provided arguments and updates the instance attributes.

        Parameters
        ----------
            ortholog (str): The ortholog identifier.
            library (str): The library identifier.
            seed (str): The seed identifier.
            tts (str): The TTS identifier.

        Returns
        -------
            None
        """
        self.BASE = self.BASE.replace(
            "ORTHOLOG", ortholog
        ).replace("LIBRARY", library)
        self.METRICS = self.BASE+self.METRICS.replace("SEED", seed).replace("TTS", tts)
        self.setups = [ortholog, library, seed, tts]
    
    def line(self):
        """
        Reads the metrics file and constructs a pandas DataFrame row.

        Reads `self.METRICS` and combines it with `self.setups` to create a single
        row containing setup details and metrics.

        Returns
        -------
            pd.DataFrame: A DataFrame with columns [ortholog, library, seed, tts,
            acc, balacc, mcc].
        """
        metrics = pd.read_csv(self.METRICS, sep="\t", index_col=None, header=None).T[0]
        line = self.setups[:]
        line.extend(metrics)
        out = pd.DataFrame(np.array(line)).T
        out.columns = self.columns
        return(out)

In [37]:
LIBRARIES = ["library1", "library2"]
SEEDS = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
ORTHOLOG_LIST = ['wt_SpCas9', 'SpCas9_VQR', 'SpCas9_EQR', 'SpCas9_VRER', 'SpCas9_D1135E', 'St1Cas9_20nt', 'St1Cas9_21nt', 'SaCas9_21nt', 'SaCas9_23nt']
TTS = ["0.01", "0.05", "0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9", "0.95", "0.99"]

In [39]:
os.makedirs("../data/processed/F2", exist_ok=True)

Read all data created by F2_gather_results.sh and concatenate it into a single dataframe, then save it to disk:

In [40]:
DIRECTORIES = itertools.product(ORTHOLOG_LIST, LIBRARIES, SEEDS, TTS)
results = []
for a in tqdm(list(DIRECTORIES)):
    try:
        r = Result()
        r.make_path(*a)
        line = r.line()
        results.append(line)
    except Exception as E:
        pass
results = pd.concat(results)
results.to_csv("../data/processed/F2a.tsv", sep="\t")

100%|████████████████████████████████████████████████████████████████████████████████████████████████| 2340/2340 [00:02<00:00, 1098.19it/s]


In [41]:
results

,ortholog,library,seed,tts,acc,balacc,mcc
0,wt_SpCas9,library1,1,0.01,0.87109375,0.4988814317673378,-0.0168691215170069
0,wt_SpCas9,library1,1,0.05,0.9453125,0.8339184305627259,0.7360380031130753
0,wt_SpCas9,library1,1,0.1,0.958984375,0.8614696265702977,0.8036680352779838
0,wt_SpCas9,library1,1,0.2,0.9599609375,0.8521683014971606,0.8082599028574968
0,wt_SpCas9,library1,1,0.3,0.95703125,0.8504904491481673,0.7932828913141632
...,...,...,...,...,...,...,...
0,SaCas9_23nt,library2,10,0.7,0.9609375,0.9075831049265584,0.8651897495842574
0,SaCas9_23nt,library2,10,0.8,0.962890625,0.9108542732831156,0.8721335637341785
0,SaCas9_23nt,library2,10,0.9,0.9609375,0.9179652310582102,0.8659266645538388
0,SaCas9_23nt,library2,10,0.95,0.962890625,0.9191599741884372,0.8725029885383222
